<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Similarity Search by Hand**


Estimated time needed: **30** minutes


In this lab, you will learn how to manually compute several common distance and similarity metrics used in machine learning and information retrieval. These metrics are fundamental to similarity search, which involves comparing data points—often represented as embedding vectors—to identify those that are most alike. Embedding vectors encode information such that their direction typically reflects the semantic content or topic of the input (e.g., a piece of text), while their magnitude can represent the intensity or importance of the encoded features. In this lab, you will calculate the Euclidean distance (L2 norm), dot product, and cosine similarity, gaining a deeper understanding of how these metrics quantify similarity and power search algorithms behind the scenes.


This completed lab includes Ashwik's fifth document and runnable solutions to all three exercises. To customize the search, edit `documents` in the embedding section and `query_text` / `top_k` in Exercise 3, then restart the kernel and run all cells. Matrix sizes are derived from the data, so adding documents does not require resizing arrays. Both documents and queries must use the same embedding model.

<h2>Table of Contents</h2>
<ol>
    <li><a href="#Similarity-Search-by-Hand">Similarity Search by Hand</a>
        <ol>
            <li><a href="#Objectives">Objectives</a></li>
            <li><a href="#Setup">Setup</a>
                <ol>
                    <li><a href="#Install-Required-Libraries">Install Required Libraries</a></li>
                    <li><a href="#Restart-the-Kernel">Restart the Kernel</a></li>
                    <li><a href="#Import-Required-Libraries">Import Required Libraries</a></li>
                </ol>
            </li>
            <li><a href="#Obtain-Vector-Embeddings">Obtain Vector Embeddings</a></li>
            <li><a href="#L2-(Euclidean)-Distance">L2 (Euclidean) Distance</a>
                <ol>
                    <li><a href="#Manual-implementation-of-L2-distance-calculation">Manual implementation of L2 distance calculation</a>
                        <ol>
                            <li><a href="#Exercise-1---Make-the-manual-calculation-more-efficient">Exercise 1 - Make the manual calculation more efficient</a></li>
                        </ol>
                    </li>
                    <li><a href="#Calculate-L2-distance-using-scipy">Calculate L2 distance using `scipy`</a></li>
                    <li><a href="#Interpret-the-L2-Distance-Results">Interpret the L2 Distance Results</a></li>
                </ol>
            </li>
            <li><a href="#Dot-Product-Similarity-and-Distance">Dot Product Similarity and Distance</a>
                <ol>
                    <li><a href="#Manual-implementation-of-dot-product-calculation">Manual implementation of dot product calculation</a></li>
                    <li><a href="#Calculate-the-dot-product-using-matrix-multiplication">Calculate the dot product using matrix multiplication</a></li>
                    <li><a href="#Calculate-dot-product-distance">Calculate dot product distance</a></li>
                </ol>
            </li>
            <li><a href="#Cosine-Similarity-and-Distance">Cosine Similarity and Distance</a>
                <ol>
                    <li><a href="#Manual-implementation-of-cosine-similarity-calculation">Manual implementation of cosine similarity calculation</a>
                        <ol>
                            <li><a href="#Calculate-the-L2-norm">Calculate the L2 norm</a></li>
                            <li><a href="#Normalize-embedding-vectors">Normalize embedding vectors</a>
                                <ol>
                                    <li><a href="#Exercise-2---Verify-that-vectors-are-normalized">Exercise 2 - Verify that vectors are normalized</a></li>
                                    <li><a href="#Normalize-embeddings-using-PyTorch">Normalize embeddings using PyTorch</a></li>
                                </ol>
                            </li>
                            <li><a href="#Calculate-cosine-similarity-manually">Calculate cosine similarity manually</a></li>
                        </ol>
                    </li>
                    <li><a href="#Calculate-cosine-similarity-using-matrix-multiplication">Calculate cosine similarity using matrix multiplication</a></li>
                    <li><a href="#Calculate-cosine-distance">Calculate cosine distance</a></li>
                </ol>
            </li>
        </ol>
    </li>
    <li><a href="#Exercise-3---Similarity-Search-Using-a-Query">Exercise 3 - Similarity Search Using a Query</a></li>
    <li><a href="#Wrap-up">Wrap-up</a></li>
    <li><a href="#Authors">Authors</a></li>
</ol>


## Objectives

After completing this lab you will be able to:

- **Compute vector similarities and distances**, including:
  - **L2 (Euclidean) distance**
  - **Dot product similarity and distance**
  - **Cosine similarity and distance**

- **Implement similarity calculations manually** using custom Python functions to reinforce core concepts.

- **Use built-in functions and methods** from external Python libraries (such as NumPy and PyTorch) to perform these computations efficiently.

- **Apply similarity measures to perform a basic similarity search**, comparing pairs of vectors to identify the most similar ones.


----


## Setup


### Install Required Libraries

From the repository root, install the notebook dependencies into your active virtual environment:

```bash
python -m pip install -r requirements-notebook.txt
jupyter lab "Similarity Search by Hand.ipynb"
```

Select that environment's Python kernel. Internet access is needed for the initial model download. If dependencies are already installed, continue to the imports.

In [1]:
# Optional: uncomment to install dependencies into the active notebook kernel.
# %pip install -r requirements-notebook.txt

### Restart the Kernel

If you installed dependencies in an already running kernel, restart it before continuing. To reproduce the lab, use **Restart Kernel and Run All Cells**. The model and all calculations are initialized by the notebook; no earlier interactive state is required.

### Import Required Libraries


For this lab, we will be using the following external libraries:


*   [`numpy`](https://numpy.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for mathematical operations.
*   [`scipy`](https://scipy.org/) for additional mathematical operations not found in `numpy`.
*   [`torch`](https://pytorch.org/) for vector operations, although this library is also typically used for deep learning.
*   [`sentence_transformers`](https://sbert.net/) for obtaining vector embeddings from text data using pre-trained models.

Run the cell below to import these libraries, along with the built-in [`math`](https://docs.python.org/3/library/math.html) module from Python’s standard library:


In [2]:
import math

import numpy as np
import scipy
import torch
from sentence_transformers import SentenceTransformer


/tmp/chromadb-notebook-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


----


## Obtain Vector Embeddings


To calculate distance and similarity metrics, we first need to generate vector embeddings for some text documents. Let’s begin by defining a few example documents:


In [3]:
# Example documents
documents = [
    'Bugs introduced by the intern had to be squashed by the lead developer.',
    'Bugs found by the quality assurance engineer were difficult to debug.',
    'Bugs are common throughout the warm summer months, according to the entomologist.',
    'Bugs, in particular spiders, are extensively studied by arachnologists.',
    'Ashwik is responsible for finding bugs in other people’s code and solving them'
    
]

There are five example documents: the first two and Ashwik's fifth sentence concern software bugs, while the third and fourth concern insects or spiders. Shared words alone do not establish the intended meaning; the surrounding context matters.

Edit `documents` to add your own examples. Keep at least two non-empty strings for the pairwise demonstrations below. Then rerun from the embedding cell onward so all matrices reflect the updated dataset.

In [4]:
# Load a pre-trained model
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

The code above creates an instance of the SentenceTransformer class from the sentence_transformers library, which is commonly used to generate vector embeddings from pre-trained models.

In this example, we’re using the [paraphrase-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/paraphrase-MiniLM-L6-v2) model. It was trained on pairs of paraphrased sentences, with the goal of generating similar embeddings for sentences that express the same meaning. While the model was originally designed for paraphrase identification, it also performs well on general semantic similarity tasks—like the ones we’re exploring in this lab.

Let's now use the model instance to encode the documents into embedding vectors:


In [5]:
# Float64 makes manual and vectorized arithmetic easier to compare numerically.
assert len(documents) >= 2 and all(isinstance(d, str) and d.strip() for d in documents)
embeddings = model.encode(documents).astype(np.float64)
n_documents = len(documents)

Let's explore the shape of our embeddings:


In [6]:
embeddings.shape

(5, 384)

And let's have a quick look at them:


In [7]:
embeddings

array([[-0.2280439 , -0.24647684, -0.00319291, ...,  0.45528126,
         0.63419694,  0.53750539],
       [-0.35791564, -0.3208397 ,  0.15963259, ..., -0.07050648,
         0.92750221,  0.34377316],
       [ 0.20302911, -0.26898569,  0.16285123, ..., -0.19651014,
        -0.03379847,  0.59561527],
       [-0.04264286, -0.45721608, -0.09526495, ..., -0.58030736,
         0.17248398,  0.09127878],
       [-0.72418708,  0.10246862, -0.26903781, ...,  0.37283596,
         0.40561715,  0.079218  ]], shape=(5, 384))

The five documents produce an array with shape `(5, 384)`: one row per document and 384 embedding features per row. If you change the documents or model, use `embeddings.shape` above to inspect the new dimensions.

## L2 (Euclidean) Distance


The L2 (Euclidean) distance between 2 vectors $a$ and $b$ can be calculated using <br>
$$ \text{L2}(a,b) = \ \sqrt{\sum_{i=1}^n (a_i - b_i)^2} $$

Let's try to implement the L2 distance manually before looking at off-the-shelf solutions available from third party libraries.


### Manual implementation of L2 distance calculation


The function below implements the L2 distance formula. It first calculates the sum of the squared differences between corresponding elements, then returns the square root of that sum:


In [8]:
def euclidean_distance_fn(vector1, vector2):
    squared_sum = sum((x - y) ** 2 for x, y in zip(vector1, vector2))
    return math.sqrt(squared_sum)

Note that `euclidean_distance_fn` computes the distance between two individual vectors, not across an entire array. To see how it works, let's calculate the distance between the first vector (index 0) and the second vector (index 1):


In [9]:
euclidean_distance_fn(embeddings[1], embeddings[0])

5.961789606546417

Of course, it does not matter which vector we put first into the function, the resulting distance is the same:


In [10]:
euclidean_distance_fn(embeddings[0], embeddings[1])

5.961789606546417

In order to calculate all of the distances between all of the vectors in the `embeddings` array, we can use nested loops. In the following, `i` and `j` are the indices of the vectors in the array: 


In [11]:
l2_dist_manual = np.zeros((n_documents, n_documents))
print(l2_dist_manual)
for i in range(embeddings.shape[0]):
    for j in range(embeddings.shape[0]):
        l2_dist_manual[i,j] = euclidean_distance_fn(embeddings[i], embeddings[j])

l2_dist_manual

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


array([[0.        , 5.96178961, 7.33940011, 7.15578276, 5.95414214],
       [5.96178961, 0.        , 7.76861572, 7.39359093, 6.65708703],
       [7.33940011, 7.76861572, 0.        , 5.91992791, 7.83335072],
       [7.15578276, 7.39359093, 5.91992791, 0.        , 6.82758616],
       [5.95414214, 6.65708703, 7.83335072, 6.82758616, 0.        ]])

`l2_dist_manual` is a 4×4 array where each element represents the L2 distance between two vectors: the vector at the row index and the vector at the column index. For example, the distance between the first vector (index 0) and the second vector (index 1) is located at position `[0, 1]` in the array:


In [12]:
l2_dist_manual[0,1]

np.float64(5.961789606546417)

Moreover, you'll notice that the upper and lower triangles of the array are identical. This is not a coincidence. Just like with the `euclidean_distance_fn` function, the distance between vectors $a$ and $b$ should be identical to the distance between vectors $b$ and $a$:


In [13]:
l2_dist_manual[1,0]

np.float64(5.961789606546417)

#### Exercise 1 - Make the manual calculation more efficient


The code used to populate the `l2_dist_manual array` is not very efficient. First, it redundantly calculates the distance between a vector and itself, even though the L2 distance in such cases is always zero. Second, the array is symmetric—meaning the distance between vectors at indices $i$ and $j$ is the same as between $j$ and $i$. Therefore, each distance only needs to be computed once.

In the cell below, write an improved version of the code that avoids these inefficiencies. The completed solution below calculates the upper triangle and mirrors each result into the lower triangle.

In [14]:
# Compute each distinct pair once and mirror it; the diagonal stays zero.
l2_dist_manual_improved = np.zeros((n_documents, n_documents))
for i in range(n_documents):
    for j in range(i + 1, n_documents):
        distance = euclidean_distance_fn(embeddings[i], embeddings[j])
        l2_dist_manual_improved[i, j] = distance
        l2_dist_manual_improved[j, i] = distance

assert np.allclose(l2_dist_manual_improved, l2_dist_manual)
l2_dist_manual_improved

array([[0.        , 5.96178961, 7.33940011, 7.15578276, 5.95414214],
       [5.96178961, 0.        , 7.76861572, 7.39359093, 6.65708703],
       [7.33940011, 7.76861572, 0.        , 5.91992791, 7.83335072],
       [7.15578276, 7.39359093, 5.91992791, 0.        , 6.82758616],
       [5.95414214, 6.65708703, 7.83335072, 6.82758616, 0.        ]])

<details>
    <summary>Click here for the solution</summary>

```python
l2_dist_manual_improved = np.zeros((n_documents, n_documents))
for i in range(embeddings.shape[0]):
    for j in range(embeddings.shape[0]):
        if j > i: # Calculate the upper triangle only
            l2_dist_manual_improved[i,j] = euclidean_distance_fn(embeddings[i], embeddings[j])
        elif i > j: # Copy the upper triangle to the lower triangle
            l2_dist_manual_improved[i,j] = l2_dist_manual[j,i]

l2_dist_manual_improved
```

</details>

### Calculate L2 distance using `scipy`


Instead of writing your own function, you can use a variety of different functions provided by external libraries to calculate the L2 distance. The example below uses a function from `scipy`: 


In [15]:
l2_dist_scipy = scipy.spatial.distance.cdist(embeddings, embeddings, 'euclidean')
l2_dist_scipy

array([[0.        , 5.96178961, 7.33940011, 7.15578276, 5.95414214],
       [5.96178961, 0.        , 7.76861572, 7.39359093, 6.65708703],
       [7.33940011, 7.76861572, 0.        , 5.91992791, 7.83335072],
       [7.15578276, 7.39359093, 5.91992791, 0.        , 6.82758616],
       [5.95414214, 6.65708703, 7.83335072, 6.82758616, 0.        ]])

The following verifies that `l2_dist_manual` and `l2_dist_scipy` are identical (after accounting for rounding errors) by using the `allclose()` function from `numpy`:


In [16]:
np.allclose(l2_dist_manual, l2_dist_scipy)

True

### Interpret the L2 Distance Results


Compare the off-diagonal values in `l2_dist_scipy` to identify the closest document pairs. Smaller Euclidean distances indicate closer embeddings. The software-related and insect-related sentences provide a useful comparison, but exact distances and rankings depend on the model and current documents. Self-distances on the diagonal are zero and should be excluded when looking for another document.

## Dot Product Similarity and Distance


The dot product between vectors $a$ and $b$ is calculated using  <br>
$$ a \cdot b = \sum_{i=1}^{n} a_i b_i $$

Let's define a custom function for calculating the dot product between two vectors.


### Manual implementation of dot product calculation


The function below implements the dot product formula:


In [17]:
def dot_product_fn(vector1, vector2):
    return sum(x * y for x, y in zip(vector1, vector2))

Let's use the dot product function to calculate the dot product between the first and second vectors:


In [18]:
dot_product_fn(embeddings[0], embeddings[1])

np.float64(18.535404087741167)

Just like we did with the L2 distance calculation, let's use nested loops to calculate the dot product between all vectors in the `embeddings` array:


In [19]:
dot_product_manual = np.empty((n_documents, n_documents))
print()
for i in range(embeddings.shape[0]):
    for j in range(embeddings.shape[0]):
        dot_product_manual[i,j] = dot_product_fn(embeddings[i], embeddings[j])

dot_product_manual

array([[33.74442073, 18.53540409,  8.56981649,  7.83093394, 15.38937391],
       [18.53540409, 38.86932276,  7.88996945,  8.66340499, 13.51932543],
       [ 8.56981649,  7.88996945, 37.26200629, 17.66956695,  4.19337929],
       [ 7.83093394,  8.66340499, 17.66956695, 33.12267404,  9.49643853],
       [15.38937391, 13.51932543,  4.19337929,  9.49643853, 32.48613577]])

The dot-product matrix is symmetric. Its diagonal contains each vector's squared norm, so self-similarity is generally non-zero. Higher dot products indicate greater similarity, but vector magnitude influences the score. Inspect the actual matrix to compare software-related documents with the insect-related documents; exact rankings depend on the model and text.

### Calculate the dot product using matrix multiplication


We can compute the dot product efficiently using the [matrix multiplication](https://en.wikipedia.org/wiki/Matrix_multiplication) operator `@`. To do this, we multiply the `embeddings` matrix by its transpose. Since `embeddings` has a shape of 4×384, its transpose will be 384×4. Multiplying these gives us a 4×4 matrix, which is the desired result:


In [20]:
# Matrix multiplication operator
dot_product_operator = embeddings @ embeddings.T
dot_product_operator

array([[33.74442073, 18.53540409,  8.56981649,  7.83093394, 15.38937391],
       [18.53540409, 38.86932276,  7.88996945,  8.66340499, 13.51932543],
       [ 8.56981649,  7.88996945, 37.26200629, 17.66956695,  4.19337929],
       [ 7.83093394,  8.66340499, 17.66956695, 33.12267404,  9.49643853],
       [15.38937391, 13.51932543,  4.19337929,  9.49643853, 32.48613577]])

We can verify that the matrix multiplication operator returns the same result as our custom function after accounting for rounding:


In [21]:
np.allclose(dot_product_manual, dot_product_operator, atol=1e-05)

True

Equivalently, if both of the matrices we want to multiply are two-dimensional, we can use the `matmul()` function from `numpy` instead:


In [22]:
# Equivalent to `np.matmul()` if both arrays are 2-D:
np.matmul(embeddings,embeddings.T)

array([[33.74442073, 18.53540409,  8.56981649,  7.83093394, 15.38937391],
       [18.53540409, 38.86932276,  7.88996945,  8.66340499, 13.51932543],
       [ 8.56981649,  7.88996945, 37.26200629, 17.66956695,  4.19337929],
       [ 7.83093394,  8.66340499, 17.66956695, 33.12267404,  9.49643853],
       [15.38937391, 13.51932543,  4.19337929,  9.49643853, 32.48613577]])

Finally, `numpy` also provides a `dot()` function, which provides an identical result:


In [23]:
# `np.dot` returns an identical result, but `np.matmul` is recommended if both arrays are 2-D:
np.dot(embeddings,embeddings.T)

array([[33.74442073, 18.53540409,  8.56981649,  7.83093394, 15.38937391],
       [18.53540409, 38.86932276,  7.88996945,  8.66340499, 13.51932543],
       [ 8.56981649,  7.88996945, 37.26200629, 17.66956695,  4.19337929],
       [ 7.83093394,  8.66340499, 17.66956695, 33.12267404,  9.49643853],
       [15.38937391, 13.51932543,  4.19337929,  9.49643853, 32.48613577]])

### Calculate dot product distance


The dot product between two vectors provides a similarity score. If, on the other hand, we would like a distance, we can simply take the negative of the dot product:


In [24]:
dot_product_distance = -dot_product_manual
dot_product_distance

array([[-33.74442073, -18.53540409,  -8.56981649,  -7.83093394,
        -15.38937391],
       [-18.53540409, -38.86932276,  -7.88996945,  -8.66340499,
        -13.51932543],
       [ -8.56981649,  -7.88996945, -37.26200629, -17.66956695,
         -4.19337929],
       [ -7.83093394,  -8.66340499, -17.66956695, -33.12267404,
         -9.49643853],
       [-15.38937391, -13.51932543,  -4.19337929,  -9.49643853,
        -32.48613577]])

Negating a dot product reverses its ranking: lower values correspond to larger dot products. This is a ranking convention rather than a mathematical distance metric; it can be negative and does not generally give zero for a vector compared with itself.

## Cosine Similarity and Distance


The cosine similarity between vectors $a$ and $b$ is calculated using <br>
$$ \text{cossim}(a, b) = \frac{a \cdot b}{||a|| \ ||b||} =  \frac{a}{||a||} \cdot \frac{b}{||b||} $$

where $||a|| = \sqrt{\sum_{k=1}^n a_k^2}$ is the L2 norm, or the magnitude, of vector $a$, and $\cdot$ is the dot product.

Also note that $\frac{a}{||a||}$ represents a normalized vector. This means it has the same direction as vector $a$ but a magnitude (or length) of 1. Thus, cosine similarity can be calculated by taking the dot product of two normalized vectors.


### Manual implementation of cosine similarity calculation


Since we’ve already covered how to compute the dot product, our strategy for manually calculating cosine similarity will focus on normalizing the vectors. This is because cosine similarity is simply the dot product of two normalized vectors, as was shown after the last equals sign in the cosine similarity calculation formula.

However, in order to normalize vectors, we must first compute their L2 norms.


#### Calculate the L2 norm


The following calculates the L2 norms for all the vectors in the `embeddings` array. The calculation simply squares each vector component, sums across columns (note the `axis=1` parameter in the `sum`), and takes a square root:


In [25]:
# L2 norms
l2_norms = np.sqrt(np.sum(embeddings**2, axis=1))
l2_norms

array([5.80899481, 6.23452667, 6.10426132, 5.75523015, 5.69966102])

Note that the result is a vector with 4 numbers, each of which corresponds to the L2 norm, or the magnitude, of each vector. In order to normalize the vectors to a length of one, we should divide each vector's components by the norm. However, in order to do that efficiently, let's reshape the `l2_norms` vector into a 4x1 array:


In [26]:
# L2 norms reshaped
l2_norms_reshaped = l2_norms.reshape(-1,1)
l2_norms_reshaped

array([[5.80899481],
       [6.23452667],
       [6.10426132],
       [5.75523015],
       [5.69966102]])

#### Normalize embedding vectors


The following code calculates normalized embedding vectors by dividing every component in the vector by the vector's L2 norm:


In [27]:
if np.any(l2_norms_reshaped == 0):
    raise ValueError("Cannot normalize a zero-length embedding.")
normalized_embeddings_manual = embeddings / l2_norms_reshaped
normalized_embeddings_manual

array([[-0.03925703, -0.04243021, -0.00054965, ...,  0.07837522,
         0.10917499,  0.09252984],
       [-0.05740863, -0.05146176,  0.02560461, ..., -0.01130904,
         0.14876867,  0.05514022],
       [ 0.03326023, -0.04406523,  0.02667829, ..., -0.03219229,
        -0.00553687,  0.09757368],
       [-0.00740941, -0.07944358, -0.01655276, ..., -0.1008313 ,
         0.02996995,  0.01586014],
       [-0.12705792,  0.01797802, -0.04720242, ...,  0.06541371,
         0.07116514,  0.01389872]], shape=(5, 384))

##### Exercise 2 - Verify that vectors are normalized


Verify that `normalized_embeddings_manual` are normalized vectors by making sure that the length of each vector is equal to 1.


In [28]:
# Exercise 2: every normalized vector should have unit length.
normalized_lengths = np.sqrt(np.sum(normalized_embeddings_manual**2, axis=1))
assert np.allclose(normalized_lengths, 1.0)
normalized_lengths

array([1., 1., 1., 1., 1.])

<details>
    <summary>Click here for the solution</summary>

```python
# Note that the length of a vector can be found by taking its L2 norm. So, to solve this exercise, you just have to calculate the L2 norm of `normalized_embeddings_manual`, and verify that the sums are equal to or very close to 1:

np.sqrt(np.sum(normalized_embeddings_manual**2, axis=1))
```
</details>


#### Normalize embeddings using PyTorch


You can normalize embeddings in PyTorch using `torch.nn.functional.normalize()`. If your data is in a NumPy array, first convert it to a PyTorch tensor using `torch.from_numpy()`. After normalization, convert the tensor back to a NumPy array using the `numpy()` method:


In [29]:
normalized_embeddings_torch = torch.nn.functional.normalize(
    torch.from_numpy(embeddings)
).numpy()
normalized_embeddings_torch

array([[-0.03925703, -0.04243021, -0.00054965, ...,  0.07837522,
         0.10917499,  0.09252984],
       [-0.05740863, -0.05146176,  0.02560461, ..., -0.01130904,
         0.14876867,  0.05514022],
       [ 0.03326023, -0.04406523,  0.02667829, ..., -0.03219229,
        -0.00553687,  0.09757368],
       [-0.00740941, -0.07944358, -0.01655276, ..., -0.1008313 ,
         0.02996995,  0.01586014],
       [-0.12705792,  0.01797802, -0.04720242, ...,  0.06541371,
         0.07116514,  0.01389872]], shape=(5, 384))

We can verify that the normalized embeddings we calculated manually and the normalized embeddings calculated using `torch` are indeed identical using numpy's `allclose()` function:


In [30]:
np.allclose(normalized_embeddings_manual, normalized_embeddings_torch)

True

#### Calculate cosine similarity manually


To calculate cosine similarity between two normalized embedding vectors, we simply take their dot product. To do this, we can leverage the dot product function we defined before. For instance, the following calculates the cosine similarity between the vector embeddings of the first and second sentence:


In [31]:
dot_product_fn(normalized_embeddings_manual[0], normalized_embeddings_manual[1])

np.float64(0.5117968198121464)

Likewise, to calculate the cosine similarities between all normalized vectors, we can use a nested loop:


In [32]:
cosine_similarity_manual = np.empty((n_documents, n_documents))
for i in range(normalized_embeddings_manual.shape[0]):
    for j in range(normalized_embeddings_manual.shape[0]):
        cosine_similarity_manual[i,j] = dot_product_fn(
            normalized_embeddings_manual[i], 
            normalized_embeddings_manual[j]
        )

cosine_similarity_manual

array([[1.        , 0.51179682, 0.24167817, 0.23423397, 0.46480515],
       [0.51179682, 1.        , 0.20731881, 0.24144735, 0.38045431],
       [0.24167817, 0.20731881, 1.        , 0.50295612, 0.12052635],
       [0.23423397, 0.24144735, 0.50295612, 1.        , 0.28950031],
       [0.46480515, 0.38045431, 0.12052635, 0.28950031, 1.        ]])

Cosine similarity ranges from -1 to 1 and the matrix is symmetric. Values closer to 1 indicate more closely aligned vectors. Unlike the raw dot product, cosine similarity removes the effect of vector magnitude. Inspect the scores rather than assuming a specific pair must rank highest.

### Calculate cosine similarity using matrix multiplication


Just like with the dot product, we can compute cosine similarity using matrix algebra. By multiplying the matrix of normalized embeddings with its transpose using the matrix multiplication operator, we obtain the cosine similarity matrix. This works because, once vectors are normalized, cosine similarity can be calculated by simply taking the dot product:


In [33]:
cosine_similarity_operator = normalized_embeddings_manual @ normalized_embeddings_manual.T
cosine_similarity_operator

array([[1.        , 0.51179682, 0.24167817, 0.23423397, 0.46480515],
       [0.51179682, 1.        , 0.20731881, 0.24144735, 0.38045431],
       [0.24167817, 0.20731881, 1.        , 0.50295612, 0.12052635],
       [0.23423397, 0.24144735, 0.50295612, 1.        , 0.28950031],
       [0.46480515, 0.38045431, 0.12052635, 0.28950031, 1.        ]])

We can verify that the matrix algebra solution is the same as the one found using the nested loop:


In [34]:
np.allclose(cosine_similarity_manual, cosine_similarity_operator)

True

### Calculate cosine distance


The cosine distance between vectors $a$ and $b$ is simply 1 minus the cosine similarity between $a$ and $b$: <br>
$$ 1 - cossim(a,b) $$

Using numpy, this can be calculated as follows:


In [35]:
1 - cosine_similarity_manual

array([[-4.44089210e-16,  4.88203180e-01,  7.58321834e-01,
         7.65766031e-01,  5.35194845e-01],
       [ 4.88203180e-01,  4.44089210e-16,  7.92681185e-01,
         7.58552655e-01,  6.19545691e-01],
       [ 7.58321834e-01,  7.92681185e-01,  3.33066907e-16,
         4.97043884e-01,  8.79473651e-01],
       [ 7.65766031e-01,  7.58552655e-01,  4.97043884e-01,
        -6.66133815e-16,  7.10499686e-01],
       [ 5.35194845e-01,  6.19545691e-01,  8.79473651e-01,
         7.10499686e-01,  3.33066907e-16]])

## Exercise 3 - Similarity Search Using a Query


Use the current `documents` list and `normalized_embeddings_manual` to search all five documents. Change `query_text` to ask a different question and `top_k` to control the number of matches. The query is embedded with the same model as the documents, normalized, and compared with every document using a dot product.

The results are ordered from highest cosine similarity to lowest. Changing the text or model can change the winning document; no particular answer is hard-coded.

In [36]:
# Customize these two values, then rerun this cell.
query_text = "Who is responsible for a coding project and fixing others' mistakes?"
top_k = 3

if not isinstance(query_text, str) or not query_text.strip():
    raise ValueError("query_text must be a non-empty string.")
if not isinstance(top_k, int) or isinstance(top_k, bool) or top_k < 1:
    raise ValueError("top_k must be a positive integer.")

query_embedding = model.encode([query_text]).astype(np.float64)
query_norm = np.linalg.norm(query_embedding, axis=1, keepdims=True)
if np.any(query_norm == 0):
    raise ValueError("Cannot normalize a zero-length query embedding.")
normalized_query_embedding = query_embedding / query_norm
cosine_similarity_q3 = normalized_embeddings_manual @ normalized_query_embedding.T
scores = cosine_similarity_q3[:, 0]
ranked_indices = np.argsort(-scores, kind="stable")[:min(top_k, n_documents)]
highest_cossim_position = int(ranked_indices[0])

print(f"Query: {query_text}\n")
for rank, index in enumerate(ranked_indices, start=1):
    print(f"{rank}. Document {index + 1} | Cosine similarity: {scores[index]:.4f}")
    print(f"   {documents[index]}")

Query: Who is responsible for a coding project and fixing others' mistakes?

1. Document 5 | Cosine similarity: 0.5430
   Ashwik is responsible for finding bugs in other people’s code and solving them
2. Document 1 | Cosine similarity: 0.3856
   Bugs introduced by the intern had to be squashed by the lead developer.
3. Document 2 | Cosine similarity: 0.2896
   Bugs found by the quality assurance engineer were difficult to debug.


The result above is calculated from the current documents and query. Higher cosine similarity is better; cosine distance is `1 - similarity`, for which lower is better. The additional Ashwik sentence participates in the ranking alongside the original four lab sentences.

In [37]:
# Numerical checks for the completed exercises.
assert embeddings.shape[0] == len(documents)
assert np.allclose(l2_dist_manual, l2_dist_scipy)
assert np.allclose(l2_dist_manual_improved, l2_dist_scipy)
assert np.allclose(dot_product_manual, dot_product_operator)
assert np.allclose(normalized_embeddings_manual, normalized_embeddings_torch)
assert np.allclose(cosine_similarity_manual, cosine_similarity_operator)
assert np.allclose(np.diag(cosine_similarity_manual), 1.0)
assert np.all(np.diff(scores[ranked_indices]) <= 0)
print("All numerical consistency checks passed.")

All numerical consistency checks passed.


## Wrap-up


In this lab, you explored how to compute L2 distance, dot product similarity and distance, as well as cosine similarity and distance between pairs of vectors for the purposes of performing a similarity search. These computations were performed both through manually defined functions and by using built-in functions and methods from external Python libraries. This helped illustrate the mathematical foundations of vector similarity and how they are implemented in practical coding scenarios.


## Authors


[Wojciech \"Victor\" Fulmyk](https://www.linkedin.com/in/wfulmyk)


## Change Log


<details>
    <summary>Click here for the changelog</summary>

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
| 2026-11-18 | 0.4 | Ashwik Raj | Completed exercises, dynamic matrix sizes, query ranking, and numerical checks |
| 2025-06-15 | 0.1 | Wojciech "Victor" Fulmyk | Initial version created |
| 2025-06-16 | 0.2 | Steve Ryan | ID review/typo fix |
| 2025-06-17 | 0.3 | Wojciech "Victor" Fulmyk | Fixed TOC and broken link |

</details>


Copyright © IBM Corporation. All rights reserved.
